# Module 4A: Price/Currency/Split Staging

This notebook creates the Pre_Regulation_Ext price/currency/split staging tables in `main.regtech_ops_stg`.

**Replicates SSIS package:** `Pre_Regulation_Ext.dtsx` (truncate/reload behavior)

**Targets created:**
- `bi_output_regtechops_reg_currencyprice_ext` — CurrencyPrice snapshot for report date window
- `bi_output_regtechops_reg_ext_dailymaxprices` — DailyMaxPrices from PriceLog history
- `bi_output_regtechops_reg_ext_currencypricemaxdatewithsplit` — CurrencyPrice with split ratios
- `bi_output_regtechops_reg_ext_historysplitratio` — History split ratio reference

**Prerequisites:** All source tables verified accessible (profiling passed 2026-06-09)

In [0]:
# Widget parameters for job/interactive use
dbutils.widgets.text("report_date", "2026-06-09", "Report Date (YYYY-MM-DD)")

report_date = dbutils.widgets.get("report_date")
print(f"Running Module 4A: Price/Currency Staging for report_date = {report_date}")

In [0]:
%sql
-- Reg_CurrencyPrice_Ext: CurrencyPrice snapshot for report-date window
-- SSIS parity: Occurred >= DATEADD(HOUR, -1, @StartDate) AND Occurred <= end of @StartDate day
-- Source: main.dealing.bronze_pricelog_history_currencyprice (preferred over Trade.CurrencyPrice)

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_currencyprice_ext
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_currencyprice_ext'
AS
WITH run_window AS (
  SELECT
    CAST('${report_date}' AS TIMESTAMP) - INTERVAL 1 HOUR AS window_start_ts,
    CAST(date_add(CAST('${report_date}' AS DATE), 1) AS TIMESTAMP) AS window_end_ts
)
SELECT
  src.CurrencyPriceID,
  src.ProviderID,
  src.InstrumentID,
  src.Bid,
  src.Ask,
  src.ValidFrom,
  src.ValidTo,
  src.OccurredOnProvider,
  src.Occurred,
  src.PriceRateID,
  src.ReceivedOnPriceServer,
  src.LiquidityAccountID,
  src.USDConversionRate,
  src.MarketPriceRateID,
  src.RateLastEx,
  src.BidSpreaded,
  src.AskSpreaded,
  src.BidMarketPriceRateID,
  src.AskMarketPriceRateID,
  src.MarkupPips,
  src.MarketReceivedTime,
  src.SkewValueBid,
  src.SkewValueAsk,
  src.SkewID,
  src.USDConversionRateBidSpreaded,
  src.USDConversionRateAskSpreaded,
  src.USDConversionPriceRateID
FROM main.dealing.bronze_pricelog_history_currencyprice src
JOIN run_window w
  ON src.Occurred >= w.window_start_ts
 AND src.Occurred <= w.window_end_ts

In [0]:
%sql
-- Reg_Ext_DailyMaxPrices: Max prices per instrument for report-date
-- Source: main.dealing.bronze_pricelog_history_currencypricemaxdate (dedicated MaxDate table)
-- Note: Uses etr_ymd partition filter to avoid Parquet schema mismatch on older partitions

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_dailymaxprices
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_dailymaxprices'
AS
SELECT
  src.*
FROM main.dealing.bronze_pricelog_history_currencypricemaxdate src
WHERE src.etr_ymd = CAST('${report_date}' AS DATE)

In [0]:
%sql
-- Reg_Ext_CurrencyPriceMaxDateWithSplit: Price with split ratio data
-- Source: main.dealing.bronze_pricelog_candles_currencypricemaxdatewithsplit (selected primary)
-- Note: This is the resolved source after candidate evaluation

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_currencypricemaxdatewithsplit
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_currencypricemaxdatewithsplit'
AS
SELECT
  src.*
FROM main.dealing.bronze_pricelog_candles_currencypricemaxdatewithsplit src
WHERE CAST(src.OccurredDate AS DATE) = CAST('${report_date}' AS DATE)

In [0]:
%sql
-- Reg_Ext_HistorySplitRatio: History split ratio reference (full snapshot, no date filter)
-- Source: main.dealing.bronze_pricelog_history_splitratio
-- SSIS parity: full table snapshot (no windowing in Pre_Regulation_Ext.dtsx for this object)

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_historysplitratio
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_historysplitratio'
AS
SELECT
  src.InstrumentID,
  src.MinDate,
  src.MaxDate,
  src.AmountRatio,
  src.IsCompletedOpenPositions,
  src.AmountRatioUnAdjusted,
  src.PriceRatio,
  src.PriceRatioUnAdjusted
FROM main.dealing.bronze_pricelog_history_splitratio src
WHERE src.IsCompletedOpenPositions = 1

In [0]:
%sql
-- Validation: row counts for all Module 4A staging tables
SELECT 
  'bi_output_regtechops_reg_currencyprice_ext' AS table_name,
  COUNT(*) AS row_count
FROM main.regtech_ops_stg.bi_output_regtechops_reg_currencyprice_ext
UNION ALL
SELECT 
  'bi_output_regtechops_reg_ext_dailymaxprices',
  COUNT(*)
FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dailymaxprices
UNION ALL
SELECT 
  'bi_output_regtechops_reg_ext_currencypricemaxdatewithsplit',
  COUNT(*)
FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_currencypricemaxdatewithsplit
UNION ALL
SELECT 
  'bi_output_regtechops_reg_ext_historysplitratio',
  COUNT(*)
FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_historysplitratio